In [1]:
import requests
import pandas as pd
import json
import time

In [2]:
metro_areas= requests.get('https://api.datausa.io/attrs/geo/?sumlevel=msa').json()
metro_areas = metro_areas['data']


In [3]:
metro_names={}
for data in metro_areas:
    metro_names[data[8]]=data[9]

In [4]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/56.0.2924.76 Safari/537.36'}
pop=requests.get('http://db.datausa.io/api/?show=geo&sumlevel=msa&required=pop&year=all',headers=headers).json()['data']
for i in pop:
    if i[1] in metro_names.keys():
        i.append(metro_names[i[1]])

In [5]:
dfpop=pd.DataFrame(pop,columns=['Year','Code','Population','Place'])
dfnames=dfpop.sort_values(by=['Year','Population']).tail(10)
dfnames.reset_index()

,index,Year,Code,Population,Place
0,2895,2016,31000US14460,4728844,"Boston-Cambridge-Newton, MA-NH Metro Area"
1,2836,2016,31000US12060,5612777,"Atlanta-Sandy Springs-Roswell, GA Metro Area"
2,3345,2016,31000US33100,5926955,"Miami-Fort Lauderdale-West Palm Beach, FL Metr..."
3,3691,2016,31000US47900,6011752,"Washington-Arlington-Alexandria, DC-VA-MD-WV"
4,3461,2016,31000US37980,6047721,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD"
5,3181,2016,31000US26420,6482592,"Houston-The Woodlands-Sugar Land, TX Metro Area"
6,3004,2016,31000US19100,6957123,"Dallas-Fort Worth-Arlington, TX"
7,2954,2016,31000US16980,9528396,"Chicago-Naperville-Elgin, IL-IN-WI Metro Area"
8,3293,2016,31000US31080,13189366,"Los Angeles-Long Beach-Anaheim, CA Metro Area"
9,3403,2016,31000US35620,20031443,"New York-Newark-Jersey City, NY-NJ-PA Metro Area"


In [6]:
mpv=requests.get('https://db.datausa.io/api/?show=geo&sumlevel=msa&required=median_property_value&year=all',headers=headers).json()['data']
commute_time=requests.get('https://datausa.io/api/data?measure=Average Commute Time&drilldowns=MSA',headers=headers).json()['data']

In [7]:
commute_mode={}
poverty={}
ownership={}
income={}
age={}
property_tax={}
citizen={}
local={}
for i in range(10):
    commute_mode[dfnames.iloc[i]['Place']]=requests.get('https://datausa.io/api/data?measure=Commute%20Means,Commute%20Means%20Moe&geo='+dfnames.iloc[i]['Code']+'&drilldowns=Group',headers=headers).json()['data']
    time.sleep(5)
    poverty[dfnames.iloc[i]['Place']]=requests.get('https://datausa.io/api/data?Geography='+dfnames.iloc[i]['Code'] +'&measure=Poverty%20Population,Poverty%20Population%20Moe&Poverty%20Status=0',headers=headers).json()['data']
    time.sleep(5)
    ownership[dfnames.iloc[i]['Place']]=requests.get('https://datausa.io/api/data?measure=Household%20Ownership,Household%20Ownership%20Moe&Geography='+dfnames.iloc[i]['Code']+'&drilldowns=Occupied%20By',headers=headers).json()['data']
    time.sleep(5)
    income[dfnames.iloc[i]['Place']]=requests.get('https://datausa.io/api/data?Geography='+dfnames.iloc[i]['Code']+'&measure=Total%20Population,Total%20Population%20MOE%20Appx,Record%20Count&drilldowns=Wage%20Bin&Workforce%20Status=true&Record%20Count%3E=5',headers=headers).json()['data']
    time.sleep(5)
    age[dfnames.iloc[i]['Place']]=requests.get('https://datausa.io/api/data?Geography='+dfnames.iloc[i]['Code']+'&measures=Birthplace,Birthplace%20Moe&drilldowns=Place%20of%20Birth,Age',headers=headers).json()['data']
    time.sleep(5)
    property_tax[dfnames.iloc[i]['Place']]=requests.get('https://datausa.io/api/data?measure=Real%20Estate%20Taxes%20by%20Mortgage,Real%20Estate%20Taxes%20by%20Mortgage%20Moe&drilldowns=Real%20Estate%20Taxes%20Paid&geo='+dfnames.iloc[i]['Code'],headers=headers).json()['data']
    time.sleep(5)
    citizen[dfnames.iloc[i]['Place']]=requests.get('https://datausa.io/api/data?measure=Citizenship%20Status&drilldowns=Citizenship&Geography='+dfnames.iloc[i]['Code'],headers=headers).json()['data']
    time.sleep(5)
    local[dfnames.iloc[i]['Place']]=requests.get('https://datausa.io/api/data?measure=Foreign-Born%20Citizens,Population&Geography='+dfnames.iloc[i]['Code'],headers=headers).json()['data']

In [157]:
dfcommute_mode=None
#convert commute mode data from dictionaries to data frame
for name in commute_mode:
    for i in range(45):
        dfcommute_mode=pd.concat([dfcommute_mode, pd.DataFrame.from_dict(commute_mode[name][i],orient='index').T ])
#remove extra columns
dfcommute_mode=dfcommute_mode.drop('ID Geography',axis=1)
dfcommute_mode=dfcommute_mode.drop('Slug Geography',axis=1)
dfcommute_mode=dfcommute_mode.drop('ID Year',axis=1)
#reindex by location and year
dfcommute_mode=dfcommute_mode.set_index(['Geography','Year'])
dfcommute_mode

ID Group           Group  \
Geography                             Year                            
Boston-Cambridge-Newton, MA-NH        2017        2  Public Transit   
                                      2017        8  Worked At Home   
                                      2017        0     Drove Alone   
                                      2017        4      Motorcycle   
                                      2017        5         Bicycle   
                                      2017        1       Carpooled   
                                      2017        7           Other   
                                      2017        3            Taxi   
                                      2017        6          Walked   
                                      2016        0     Drove Alone   
                                      2016        2  Public Transit   
                                      2016        3            Taxi   
                                      2016        7           Other   
                                      2016        6          Walked   
                                      2016        1       Carpooled   
                                      2016        4      Motorcycle   
                                      2016        8  Worked At Home   
                                      2016        5         Bicycle   
                                      2015        1       Carpooled   
                                      2015        2  Public Transit   
                                      2015        0     Drove Alone   
                                      2015        6          Walked   
                                      2015        4      Motorcycle   
                                      2015        7           Other   
                                      2015        8  Worked At Home   
                                      2015        3            Taxi   
                                      2015        5         Bicycle   
                                      2014        7           Other   
                                      2014        5         Bicycle   
                                      2014        0     Drove Alone   
...                                             ...             ...   
New York-Newark-Jersey City, NY-NJ-PA 2016        4      Motorcycle   
                                      2016        8  Worked At Home   
                                      2016        5         Bicycle   
                                      2015        1       Carpooled   
                                      2015        2  Public Transit   
                                      2015        0     Drove Alone   
                                      2015        6          Walked   
                                      2015        4      Motorcycle   
                                      2015        7           Other   
                                      2015        8  Worked At Home   
                                      2015        3            Taxi   
                                      2015        5         Bicycle   
                                      2014        7           Other   
                                      2014        5         Bicycle   
                                      2014        0     Drove Alone   
                                      2014        1       Carpooled   
                                      2014        6          Walked   
                                      2014        4      Motorcycle   
                                      2014        8  Worked At Home   
                                      2014        3            Taxi   
                                      2014        2  Public Transit   
                                      2013        5         Bicycle   
                                      2013        3            Taxi   
                                      2013        7           Other

In [105]:
dfpoverty=None
for name in poverty:
    for i in range(5):
        dfpoverty=pd.concat([dfpoverty, pd.DataFrame.from_dict(poverty[name][i],orient='index').T ])
dfpoverty=dfpoverty.drop('ID Geography',axis=1)
dfpoverty=dfpoverty.drop('Slug Geography',axis=1)
dfpoverty=dfpoverty.drop('ID Year',axis=1)
#reindex by location and year
dfpoverty=dfpoverty.set_index(['Geography','Year'])
dfpoverty

ID Group           Group  \
Geography                             Year                            
Boston-Cambridge-Newton, MA-NH        2017        2  Public Transit   
                                      2017        8  Worked At Home   
                                      2017        0     Drove Alone   
                                      2017        4      Motorcycle   
                                      2017        5         Bicycle   
                                      2017        1       Carpooled   
                                      2017        7           Other   
                                      2017        3            Taxi   
                                      2017        6          Walked   
                                      2016        0     Drove Alone   
                                      2016        2  Public Transit   
                                      2016        3            Taxi   
                                      2016        7           Other   
                                      2016        6          Walked   
                                      2016        1       Carpooled   
                                      2016        4      Motorcycle   
                                      2016        8  Worked At Home   
                                      2016        5         Bicycle   
                                      2015        1       Carpooled   
                                      2015        2  Public Transit   
                                      2015        0     Drove Alone   
                                      2015        6          Walked   
                                      2015        4      Motorcycle   
                                      2015        7           Other   
                                      2015        8  Worked At Home   
                                      2015        3            Taxi   
                                      2015        5         Bicycle   
                                      2014        7           Other   
                                      2014        5         Bicycle   
                                      2014        0     Drove Alone   
...                                             ...             ...   
New York-Newark-Jersey City, NY-NJ-PA 2016        4      Motorcycle   
                                      2016        8  Worked At Home   
                                      2016        5         Bicycle   
                                      2015        1       Carpooled   
                                      2015        2  Public Transit   
                                      2015        0     Drove Alone   
                                      2015        6          Walked   
                                      2015        4      Motorcycle   
                                      2015        7           Other   
                                      2015        8  Worked At Home   
                                      2015        3            Taxi   
                                      2015        5         Bicycle   
                                      2014        7           Other   
                                      2014        5         Bicycle   
                                      2014        0     Drove Alone   
                                      2014        1       Carpooled   
                                      2014        6          Walked   
                                      2014        4      Motorcycle   
                                      2014        8  Worked At Home   
                                      2014        3            Taxi   
                                      2014        2  Public Transit   
                                      2013        5         Bicycle   
                                      2013        3            Taxi   
                                      2013        7           Other

In [121]:
dfownership=None
for name in ownership:
    for i in range(10):
        dfownership=pd.concat([dfownership, pd.DataFrame.from_dict(ownership[name][i],orient='index').T ])
dfownership=dfownership.drop('ID Geography',axis=1)
dfownership=dfownership.drop('Slug Geography',axis=1)
dfownership=dfownership.drop('ID Year',axis=1)
#reindex by location and year
dfownership=dfownership.set_index(['Geography','Year'])
dfownership

ID Occupied By  \
Geography                                 Year                  
Boston-Cambridge-Newton, MA-NH            2017              0   
                                          2017              1   
                                          2016              0   
                                          2016              1   
                                          2015              0   
                                          2015              1   
                                          2014              0   
                                          2014              1   
                                          2013              0   
                                          2013              1   
Atlanta-Sandy Springs-Roswell, GA         2017              0   
                                          2017              1   
                                          2016              0   
                                          2016              1   
                                          2015              0   
                                          2015              1   
                                          2014              0   
                                          2014              1   
                                          2013              0   
                                          2013              1   
Miami-Fort Lauderdale-West Palm Beach, FL 2017              0   
                                          2017              1   
                                          2016              0   
                                          2016              1   
                                          2015              0   
                                          2015              1   
                                          2014              0   
                                          2014              1   
                                          2013              0   
                                          2013              1   
...                                                       ...   
Chicago-Naperville-Elgin, IL-IN-WI        2017              0   
                                          2017              1   
                                          2016              0   
                                          2016              1   
                                          2015              0   
                                          2015              1   
                                          2014              0   
                                          2014              1   
                                          2013              0   
                                          2013              1   
Los Angeles-Long Beach-Anaheim, CA        2017              0   
                                          2017              1   
                                          2016              0   
                                          2016              1   
                                          2015              0   
                                          2015              1   
                                          2014              0   
                                          2014              1   
                                          2013              0   
                                          2013              1   
New York-Newark-Jersey City, NY-NJ-PA     2017              0   
                                          2017              1   
                                          2016              0   
                                          2016              1   
                                          2015              0   
                                          2015              1   
                                          2014              0   
                                          2014              1   
                                          2013              0   
                 

In [135]:
dfincome=None
for name in income:
    for i in range(84):
        dfincome=pd.concat([dfincome, pd.DataFrame.from_dict(income[name][i],orient='index').T ])
dfincome=dfincome.drop('ID Geography',axis=1)
#dfincome=dfincome.drop('Slug Geography',axis=1)
dfincome=dfincome.drop('ID Year',axis=1)
#reindex by location and year
dfincome=dfincome.set_index(['Geography','Year'])
dfincome

ID Wage Bin   Wage Bin ID Workforce Status  \
Geography     Year                                              
Massachusetts 2017          16  $150-160k                True   
              2017          21     $200k+                True   
              2017          20  $190-200k                True   
              2017           2    $10-20k                True   
              2017          19  $180-190k                True   
              2017           3    $20-30k                True   
              2017          18  $170-180k                True   
              2017           4    $30-40k                True   
              2017          17  $160-170k                True   
              2017           5    $40-50k                True   
              2017           1     < $10K                True   
              2017           6    $50-60k                True   
              2017          15  $140-150k                True   
              2017           7    $60-70k                True   
              2017          14  $130-140k                True   
              2017           8    $70-80k                True   
              2017          13  $120-130k                True   
              2017           9    $80-90k                True   
              2017          12  $110-120k                True   
              2017          10   $90-100k                True   
              2017          11  $100-110k                True   
              2016           3    $20-30k                True   
              2016           6    $50-60k                True   
              2016           1     < $10K                True   
              2016          16  $150-160k                True   
              2016          19  $180-190k                True   
              2016           7    $60-70k                True   
              2016           2    $10-20k                True   
              2016          15  $140-150k                True   
              2016           4    $30-40k                True   
...                        ...        ...                 ...   
New York      2015           4    $30-40k                True   
              2015           8    $70-80k                True   
              2015          18  $170-180k                True   
              2015          15  $140-150k                True   
              2015           5    $40-50k                True   
              2015           7    $60-70k                True   
              2015          17  $160-170k                True   
              2015          16  $150-160k                True   
              2015          11  $100-110k                True   
              2014          14  $130-140k                True   
              2014           7    $60-70k                True   
              2014          16  $150-160k                True   
              2014           6    $50-60k                True   
              2014           8    $70-80k                True   
              2014          18  $170-180k                True   
              2014          15  $140-150k                True   
              2014           5    $40-50k                True   
              2014           9    $80-90k                True   
              2014          19  $180-190k                True   
              2014          17  $160-170k                True   
              2014           4    $30-40k                True   
              2014          10   $90-100k                True   
              2014          20  $190-200k                True   
              2014          13  $120-130k                True   
              2014           3    $20-30k                True   
              2014          11  $100-110k                True   
              2014          21     $200k+                True   
              2014          12  $110-120k                True   
              2014           2    $10-20k             

In [148]:
dfage=None
for name in age:
    for i in range(220):
        dfage=pd.concat([dfage, pd.DataFrame.from_dict(age[name][i],orient='index').T ])
dfage=dfage.drop('ID Geography',axis=1)
dfage=dfage.drop('Slug Geography',axis=1)
dfage=dfage.drop('ID Year',axis=1)
#reindex by location and year
dfage=dfage.set_index(['Geography','Year'])
dfage

ID Place of Birth  \
Geography                             Year                     
Boston-Cambridge-Newton, MA-NH        2017                 2   
                                      2017                 3   
                                      2017                 3   
                                      2017                 0   
                                      2017                 3   
                                      2017                 0   
                                      2017                 3   
                                      2017                 0   
                                      2017                 3   
                                      2017                 0   
                                      2017                 3   
                                      2017                 0   
                                      2017                 3   
                                      2017                 0   
                                      2017                 3   
                                      2017                 0   
                                      2017                 3   
                                      2017                 0   
                                      2017                 3   
                                      2017                 0   
                                      2017                 3   
                                      2017                 0   
                                      2017                 0   
                                      2017                 1   
                                      2017                 2   
                                      2017                 1   
                                      2017                 2   
                                      2017                 1   
                                      2017                 2   
                                      2017                 1   
...                                                      ...   
New York-Newark-Jersey City, NY-NJ-PA 2013                 3   
                                      2013                 0   
                                      2013                 1   
                                      2013                 2   
                                      2013                 2   
                                      2013                 3   
                                      2013                 1   
                                      2013                 0   
                                      2013                 1   
                                      2013                 2   
                                      2013                 3   
                                      2013                 0   
                                      2013                 2   
                                      2013                 1   
                                      2013                 0   
                                      2013                 3   
                                      2013                 3   
                                      2013                 1   
                                      2013                 0   
                                      2013                 2   
                                      2013                 3   
                                      2013                 1   
                                      2013                 3   
                                      2013                 1   
                                      2013                 0   
                                      2013                 3   
                                      2013                 1   
                                      2013                 2   
                                      2013                 3   
                                      2013                 2   

          

In [156]:
dfproperty_tax=None
for name in property_tax:
    for i in range(30):
        dfproperty_tax=pd.concat([dfproperty_tax, pd.DataFrame.from_dict(property_tax[name][i],orient='index').T ])
dfproperty_tax=dfproperty_tax.drop('ID Geography',axis=1)
dfproperty_tax=dfproperty_tax.drop('Slug Geography',axis=1)
dfproperty_tax=dfproperty_tax.drop('ID Year',axis=1)
#reindex by location and year
dfproperty_tax=dfproperty_tax.set_index(['Geography','Year'])
dfproperty_tax

ID Real Estate Taxes Paid  \
Geography                             Year                             
Boston-Cambridge-Newton, MA-NH        2017                         5   
                                      2017                         0   
                                      2017                         4   
                                      2017                         1   
                                      2017                         3   
                                      2017                         2   
                                      2016                         1   
                                      2016                         5   
                                      2016                         0   
                                      2016                         3   
                                      2016                         2   
                                      2016                         4   
                                      2015                         4   
                                      2015                         0   
                                      2015                         3   
                                      2015                         2   
                                      2015                         5   
                                      2015                         1   
                                      2014                         3   
                                      2014                         4   
                                      2014                         1   
                                      2014                         5   
                                      2014                         2   
                                      2014                         0   
                                      2013                         4   
                                      2013                         5   
                                      2013                         0   
                                      2013                         1   
                                      2013                         2   
                                      2013                         3   
...                                                              ...   
New York-Newark-Jersey City, NY-NJ-PA 2017                         5   
                                      2017                         0   
                                      2017                         4   
                                      2017                         1   
                                      2017                         3   
                                      2017                         2   
                                      2016                         1   
                                      2016                         5   
                                      2016                         0   
                                      2016                         3   
                                      2016                         2   
                                      2016                         4   
                                      2015                         4   
                                      2015                         0   
                                      2015                         3   
                                      2015                         2   
                                      2015                         5   
                                      2015                         1   
                                      2014                         3   
                                      2014                         4   
                                      2014                         1   
                                      2014                         5   
                                      2014                         2   
           

In [161]:
dfcitizen=None
for name in citizen:
    for i in range(10):
        dfcitizen=pd.concat([dfcitizen, pd.DataFrame.from_dict(citizen[name][i],orient='index').T ])
dfcitizen=dfcitizen.drop('ID Geography',axis=1)
dfcitizen=dfcitizen.drop('Slug Geography',axis=1)
dfcitizen=dfcitizen.drop('ID Year',axis=1)
#reindex by location and year
dfcitizen=dfcitizen.set_index(['Geography','Year'])
dfcitizen

ID Citizenship  Citizenship  \
Geography                                 Year                               
Boston-Cambridge-Newton, MA-NH            2017              0      Citizen   
                                          2017              1  Non-Citizen   
                                          2016              0      Citizen   
                                          2016              1  Non-Citizen   
                                          2015              0      Citizen   
                                          2015              1  Non-Citizen   
                                          2014              0      Citizen   
                                          2014              1  Non-Citizen   
                                          2013              0      Citizen   
                                          2013              1  Non-Citizen   
Atlanta-Sandy Springs-Roswell, GA         2017              0      Citizen   
                                          2017              1  Non-Citizen   
                                          2016              0      Citizen   
                                          2016              1  Non-Citizen   
                                          2015              0      Citizen   
                                          2015              1  Non-Citizen   
                                          2014              0      Citizen   
                                          2014              1  Non-Citizen   
                                          2013              0      Citizen   
                                          2013              1  Non-Citizen   
Miami-Fort Lauderdale-West Palm Beach, FL 2017              0      Citizen   
                                          2017              1  Non-Citizen   
                                          2016              0      Citizen   
                                          2016              1  Non-Citizen   
                                          2015              0      Citizen   
                                          2015              1  Non-Citizen   
                                          2014              0      Citizen   
                                          2014              1  Non-Citizen   
                                          2013              0      Citizen   
                                          2013              1  Non-Citizen   
...                                                       ...          ...   
Chicago-Naperville-Elgin, IL-IN-WI        2017              0      Citizen   
                                          2017              1  Non-Citizen   
                                          2016              0      Citizen   
                                          2016              1  Non-Citizen   
                                          2015              0      Citizen   
                                          2015              1  Non-Citizen   
                                          2014              0      Citizen   
                                          2014              1  Non-Citizen   
                                          2013              0      Citizen   
                                          2013              1  Non-Citizen   
Los Angeles-Long Beach-Anaheim, CA        2017              0      Citizen   
                                          2017              1  Non-Citizen   
                                          2016              0      Citizen   
                                          2016              1  Non-Citizen   
                                          2015              0      Citizen   
                                          2015              1  Non-Citizen   
                                          2014              0      Citizen   
                                          2014              1  Non-Citizen   
                                          2013              0      Ci

In [164]:
dflocal=None
for name in local:
    for i in range(5):
        dflocal=pd.concat([dflocal, pd.DataFrame.from_dict(local[name][i],orient='index').T ])
dflocal=dflocal.drop('ID Geography',axis=1)
dflocal=dflocal.drop('Slug Geography',axis=1)
dflocal=dflocal.drop('ID Year',axis=1)
#reindex by location and year
dflocal=dflocal.set_index(['Geography','Year'])
dflocal

C:\Users\aleja\Anaconda3\lib\site-packages\ipykernel_launcher.py:4: FutureWarning: Sorting because non-concatenation axis is not aligned. A future version
of pandas will change to not sort by default.

To accept the future behavior, pass 'sort=False'.

To retain the current behavior and silence the warning, pass 'sort=True'.

  after removing the cwd from sys.path.


Foreign-Born Citizens  \
Geography                                    Year                         
Boston-Cambridge-Newton, MA-NH               2017                869620   
                                             2016                838257   
                                             2015                816401   
                                             2014                   NaN   
                                             2013                   NaN   
Atlanta-Sandy Springs-Roswell, GA            2017                777555   
                                             2016                753532   
                                             2015                743009   
                                             2014                   NaN   
                                             2013                   NaN   
Miami-Fort Lauderdale-West Palm Beach, FL    2017               2406964   
                                             2016               2334438   
                                             2015               2278878   
                                             2014                   NaN   
                                             2013                   NaN   
Washington-Arlington-Alexandria, DC-VA-MD-WV 2017               1377375   
                                             2016               1340621   
                                             2015               1315879   
                                             2014                   NaN   
                                             2013                   NaN   
Philadelphia-Camden-Wilmington, PA-NJ-DE-MD  2017                638538   
                                             2016                622091   
                                             2015                609314   
                                             2014                   NaN   
                                             2013                   NaN   
Houston-The Woodlands-Sugar Land, TX         2017               1538097   
                                             2016               1485232   
                                             2015               1438555   
                                             2014                   NaN   
                                             2013                   NaN   
Dallas-Fort Worth-Arlington, TX              2017               1285103   
                                             2016               1238622   
                                             2015               1206741   
                                             2014                   NaN   
                                             2013                   NaN   
Chicago-Naperville-Elgin, IL-IN-WI           2017               1689797   
                                             2016               1682220   
                                             2015               1693807   
                                             2014                   NaN   
                                             2013                   NaN   
Los Angeles-Long Beach-Anaheim, CA           2017               4433826   
                                             2016               4425953   
                                             2015               4436626   
                                             2014                   NaN   
                                             2013                   NaN   
New York-Newark-Jersey City, NY-NJ-PA        2017               5825675   
                                             2016               5749207   
                                             2015               5720729   
                                             2014                   NaN   
                                             2013                   NaN   

                                                  Population  
Geography                                    Year             
Boston-Cambridge-Newton